# Customer Segmentation & Churn Pattern Analytics in European Banking

**Project:** Customer Segmentation & Churn Pattern Analytics in European Banking  
**Dataset:** `European_Bank(1).csv`

This notebook covers **Phase 1: Data Ingestion, Validation, Cleaning, Segmentation, EDA and KPI analysis**.

The target variable is `Exited`:
- `0` = customer retained
- `1` = customer churned

We will later use the cleaned data for machine-learning churn prediction and the Streamlit dashboard.


In [ ]:
# 1. Import libraries
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report, confusion_matrix, ConfusionMatrixDisplay,
    roc_auc_score
)

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")


In [ ]:
# 2. Load the dataset
DATA_PATH = "European_Bank.csv"
df = pd.read_csv(DATA_PATH)

print("Dataset shape:", df.shape)
display(df.head())


In [ ]:
# 3. Basic validation
print("Column names:")
print(df.columns.tolist())

print("\nData types:")
display(df.dtypes.to_frame("dtype"))

print("\nMissing values:")
display(df.isnull().sum().to_frame("missing_values"))

print("\nDuplicate rows:", df.duplicated().sum())
print("Duplicate CustomerId values:", df["CustomerId"].duplicated().sum())

print("\nTarget distribution:")
display(df["Exited"].value_counts().rename(index={0: "Retained", 1: "Churned"}))


In [ ]:
# 4. Clean the data
# Keep Surname out of analytical/model features because it is not a useful behavioral variable.
# CustomerId is retained for identification but excluded from model features.
# Year is checked and can be excluded from the churn model if it has only one value.

df_clean = df.copy()

print("Unique years:", df_clean["Year"].unique())
print("Unique countries:", df_clean["Geography"].unique())

# Basic consistency checks
binary_columns = ["HasCrCard", "IsActiveMember", "Exited"]
for col in binary_columns:
    print(f"{col} unique values:", sorted(df_clean[col].dropna().unique()))


## 5. Create segmentation fields

The project requirements specify these segmentation dimensions:
- Geography: France, Spain, Germany
- Age: `<30`, `30–45`, `46–60`, `60+`
- Credit score: Low, Medium, High
- Tenure: New, Mid-term, Long-term
- Balance: Zero-balance, Low-balance, High-balance

The exact numerical cut-offs for credit score and balance are made explicit below so the analysis is reproducible.


In [ ]:
# 5A. Age segments
df_clean["AgeGroup"] = pd.cut(
    df_clean["Age"],
    bins=[-np.inf, 29, 45, 60, np.inf],
    labels=["<30", "30–45", "46–60", "60+"]
)

# 5B. Credit score bands
# Low < 580, Medium 580–669, High >= 670
df_clean["CreditScoreBand"] = pd.cut(
    df_clean["CreditScore"],
    bins=[-np.inf, 579, 669, np.inf],
    labels=["Low", "Medium", "High"]
)

# 5C. Tenure groups
# New: 0–2 years, Mid-term: 3–6 years, Long-term: 7+ years
df_clean["TenureGroup"] = pd.cut(
    df_clean["Tenure"],
    bins=[-np.inf, 2, 6, np.inf],
    labels=["New", "Mid-term", "Long-term"]
)

# 5D. Balance segments
# Zero = exactly 0; among positive balances, split into low/high using the median.
positive_balance_median = df_clean.loc[df_clean["Balance"] > 0, "Balance"].median()

df_clean["BalanceSegment"] = np.select(
    [
        df_clean["Balance"].eq(0),
        df_clean["Balance"].gt(0) & df_clean["Balance"].le(positive_balance_median),
        df_clean["Balance"].gt(positive_balance_median)
    ],
    [
        "Zero-balance",
        "Low-balance",
        "High-balance"
    ],
    default="Unknown"
)

print("Positive-balance median used:", positive_balance_median)
display(df_clean[[
    "Geography", "Age", "AgeGroup", "CreditScore", "CreditScoreBand",
    "Tenure", "TenureGroup", "Balance", "BalanceSegment"
]].head(10))


## 6. KPI calculations

Required KPIs:
1. Overall Churn Rate
2. Segment Churn Rate
3. High-Value Churn Ratio
4. Geographic Risk Index
5. Engagement Drop Indicator

For this project, **high-value customers** are defined as customers in the `High-balance` segment.

The **Geographic Risk Index** is defined as each country's churn rate divided by the overall churn rate. A value above 1 means the country has higher-than-average churn risk.

The **Engagement Drop Indicator** compares churn among inactive members with churn among active members.


In [ ]:
# 6A. Overall churn rate
overall_churn_rate = df_clean["Exited"].mean() * 100

# 6B. Segment churn rate helper
def segment_churn_rate(data, column):
    result = (
        data.groupby(column, observed=False)["Exited"]
        .agg(["count", "sum", "mean"])
        .rename(columns={"count": "Customers", "sum": "Churned", "mean": "ChurnRate"})
    )
    result["ChurnRate"] = result["ChurnRate"] * 100
    result["ChurnContribution"] = result["Churned"] / result["Churned"].sum() * 100
    return result.sort_values("ChurnRate", ascending=False)

# 6C. High-value churn ratio
high_value = df_clean[df_clean["BalanceSegment"] == "High-balance"]
high_value_churn_ratio = high_value["Exited"].mean() * 100

# 6D. Geographic Risk Index
geo = segment_churn_rate(df_clean, "Geography")
geo["GeographicRiskIndex"] = geo["ChurnRate"] / overall_churn_rate

# 6E. Engagement Drop Indicator
inactive_churn_rate = df_clean.loc[df_clean["IsActiveMember"] == 0, "Exited"].mean() * 100
active_churn_rate = df_clean.loc[df_clean["IsActiveMember"] == 1, "Exited"].mean() * 100
engagement_drop_indicator = inactive_churn_rate - active_churn_rate

kpis = pd.DataFrame({
    "KPI": [
        "Overall Churn Rate",
        "High-Value Churn Ratio",
        "Inactive Customer Churn Rate",
        "Active Customer Churn Rate",
        "Engagement Drop Indicator"
    ],
    "Value": [
        overall_churn_rate,
        high_value_churn_ratio,
        inactive_churn_rate,
        active_churn_rate,
        engagement_drop_indicator
    ]
})

display(kpis)

print("Geographic Risk Index:")
display(geo)


## 7. Churn distribution analysis

We now examine churn across the required segmentation dimensions.


In [ ]:
# Geography
display(segment_churn_rate(df_clean, "Geography"))

# Age
display(segment_churn_rate(df_clean, "AgeGroup"))

# Credit score
display(segment_churn_rate(df_clean, "CreditScoreBand"))

# Tenure
display(segment_churn_rate(df_clean, "TenureGroup"))

# Balance
display(segment_churn_rate(df_clean, "BalanceSegment"))

# Gender
display(segment_churn_rate(df_clean, "Gender"))


In [ ]:
# 8. Churned vs retained customer profiles
profile_columns = [
    "CreditScore", "Age", "Tenure", "Balance",
    "NumOfProducts", "EstimatedSalary", "IsActiveMember", "HasCrCard"
]

profile_comparison = df_clean.groupby("Exited")[profile_columns].mean().T
profile_comparison.columns = ["Retained (0)", "Churned (1)"]

display(profile_comparison)


## 9. Visualizations


In [ ]:
# Helper function for churn-rate bar charts
def plot_churn_rate(column, title):
    rates = df_clean.groupby(column, observed=False)["Exited"].mean().mul(100).sort_values(ascending=False)

    plt.figure(figsize=(8, 5))
    rates.plot(kind="bar")
    plt.title(title)
    plt.ylabel("Churn Rate (%)")
    plt.xlabel(column)
    plt.xticks(rotation=0)
    plt.tight_layout()
    plt.show()

plot_churn_rate("Geography", "Churn Rate by Geography")


In [ ]:
plot_churn_rate("AgeGroup", "Churn Rate by Age Group")


In [ ]:
plot_churn_rate("TenureGroup", "Churn Rate by Tenure Group")


In [ ]:
plot_churn_rate("CreditScoreBand", "Churn Rate by Credit Score Band")


In [ ]:
plot_churn_rate("BalanceSegment", "Churn Rate by Balance Segment")


In [ ]:
plot_churn_rate("Gender", "Churn Rate by Gender")


In [ ]:
# Geography × Age interaction
geo_age = (
    df_clean.groupby(["Geography", "AgeGroup"], observed=False)["Exited"]
    .mean()
    .mul(100)
    .reset_index(name="ChurnRate")
)

display(geo_age.sort_values("ChurnRate", ascending=False))


In [ ]:
# High-value customer analysis
high_value_summary = (
    df_clean.assign(HighValue=df_clean["BalanceSegment"].eq("High-balance"))
    .groupby("HighValue")["Exited"]
    .agg(["count", "sum", "mean"])
    .rename(columns={"count": "Customers", "sum": "Churned", "mean": "ChurnRate"})
)

high_value_summary["ChurnRate"] *= 100
display(high_value_summary)


## 10. Prepare modeling data

For the ML model:
- `Exited` is the target.
- `Surname` is excluded.
- `CustomerId` is excluded because it is only an identifier.
- `Year` is excluded from the predictive model when it contains a single year.
- The engineered segmentation columns are useful for analytics, while the original numeric/categorical customer attributes are used as model inputs.


In [ ]:
# Model dataset
model_features = [
    "CreditScore", "Geography", "Gender", "Age", "Tenure",
    "Balance", "NumOfProducts", "HasCrCard",
    "IsActiveMember", "EstimatedSalary"
]

X = df_clean[model_features]
y = df_clean["Exited"]

categorical_features = ["Geography", "Gender"]
numeric_features = [
    "CreditScore", "Age", "Tenure", "Balance",
    "NumOfProducts", "HasCrCard", "IsActiveMember", "EstimatedSalary"
]

preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numeric_features),
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features)
    ]
)

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training rows:", len(X_train))
print("Testing rows:", len(X_test))


## 11. Machine-learning models

We compare:
1. Logistic Regression — interpretable baseline.
2. Random Forest — non-linear ensemble model.

The next section evaluates accuracy, precision, recall, F1-score and ROC-AUC.


In [ ]:
# Logistic Regression
logistic_model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(max_iter=1000, class_weight="balanced"))
])

logistic_model.fit(X_train, y_train)
logistic_pred = logistic_model.predict(X_test)
logistic_prob = logistic_model.predict_proba(X_test)[:, 1]

# Random Forest
rf_preprocessor = ColumnTransformer(
    transformers=[
        ("num", "passthrough", numeric_features),
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features)
    ]
)

rf_model = Pipeline(steps=[
    ("preprocessor", rf_preprocessor),
    ("classifier", RandomForestClassifier(
        n_estimators=300,
        random_state=42,
        class_weight="balanced",
        max_depth=None
    ))
])

rf_model.fit(X_train, y_train)
rf_pred = rf_model.predict(X_test)
rf_prob = rf_model.predict_proba(X_test)[:, 1]

def evaluate_model(name, y_true, pred, prob):
    return {
        "Model": name,
        "Accuracy": accuracy_score(y_true, pred),
        "Precision": precision_score(y_true, pred, zero_division=0),
        "Recall": recall_score(y_true, pred, zero_division=0),
        "F1": f1_score(y_true, pred, zero_division=0),
        "ROC_AUC": roc_auc_score(y_true, prob)
    }

model_results = pd.DataFrame([
    evaluate_model("Logistic Regression", y_test, logistic_pred, logistic_prob),
    evaluate_model("Random Forest", y_test, rf_pred, rf_prob)
])

display(model_results)


In [ ]:
# Detailed reports
print("LOGISTIC REGRESSION")
print(classification_report(y_test, logistic_pred, target_names=["Retained", "Churned"]))

print("RANDOM FOREST")
print(classification_report(y_test, rf_pred, target_names=["Retained", "Churned"]))


In [ ]:
# Confusion matrix for the selected model
# Change selected_model_pred after reviewing model_results.
selected_model_name = model_results.sort_values("F1", ascending=False).iloc[0]["Model"]
selected_model_pred = rf_pred if selected_model_name == "Random Forest" else logistic_pred

cm = confusion_matrix(y_test, selected_model_pred)
disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=["Retained", "Churned"]
)
disp.plot()
plt.title(f"Confusion Matrix — {selected_model_name}")
plt.tight_layout()
plt.show()

print("Selected model based on F1-score:", selected_model_name)


In [ ]:
# 12. Save the cleaned dataset for later Streamlit work
OUTPUT_PATH = "european_bank_cleaned.csv"
df_clean.to_csv(OUTPUT_PATH, index=False)

print(f"Saved cleaned dataset to: {OUTPUT_PATH}")
